In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import platform
import warnings
warnings.filterwarnings('ignore')

# -----------------------------------------------------------
# [기본 설정] 맥/윈도우 폰트 깨짐 자동 방지
# -----------------------------------------------------------
if platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')
else:
    plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

# -----------------------------------------------------------
# [데이터 불러오기] 업로드된 상세현황 파일명으로 직접 지정
# -----------------------------------------------------------
file_acc = 'DB_Accidents_Detailed_2022_2024.csv'

if not os.path.exists(file_acc):
    print(f"🚨 '{file_acc}' 파일을 찾을 수 없습니다. 현재 폴더를 확인해주세요.")
else:
    # 인코딩 처리하며 파일 읽기
    try:
        df_acc = pd.read_csv(file_acc, encoding='utf-8')
    except UnicodeDecodeError:
        df_acc = pd.read_csv(file_acc, encoding='cp949')
        
    # 컬럼명 띄어쓰기 정리 (안전장치)
    df_acc.columns = df_acc.columns.str.strip()
        
    # -----------------------------------------------------------
    # [데이터 분석] 사고 원인별 발생 건수 집계 및 1% 미만 통합
    # -----------------------------------------------------------
    cause_counts = df_acc['원인'].value_counts()
    total_accidents = cause_counts.sum()
    
    # 1%에 해당하는 기준 건수 계산
    threshold = total_accidents * 0.01
    
    # 1% 이상인 주요 원인과 1% 미만인 소수 원인 분리
    main_causes = cause_counts[cause_counts >= threshold].copy()
    minor_causes_sum = cause_counts[cause_counts < threshold].sum()
    
    # 소수 원인들의 합을 '기타'에 더해주기 (기존에 '기타'가 있으면 합치고, 없으면 새로 생성)
    if minor_causes_sum > 0:
        if '기타' in main_causes.index:
            main_causes['기타'] += minor_causes_sum
        else:
            main_causes['기타'] = minor_causes_sum

    # 다시 크기순으로 정렬
    main_causes = main_causes.sort_values(ascending=False)
    
